# Multi-constellation FSPL dataset generation

Version 1: A is the victim ground station, connected to one A satellite; B is the aggressor ground station, connected to one B satellite. The co-channel downlink from B's active satellite interferes at A. The model uses only FSPL, identical fixed transmit power/gain, fixed receive gain, and thermal noise. **Positions come from orbit CSVs; communication behavior is simulated.**

Read orbits from CSV files without network access or repeated propagation. See `README.md` for input requirements, scenario limits, and handover scoring.


In [ ]:
from pathlib import Path
from dataclasses import replace
import sys
import json
import pandas as pd
from IPython.display import display

ROOT = next((p for base in [Path.cwd(), *Path.cwd().parents]
             for p in [base, base / "generate data"]
             if (p / "dataset.py").is_file() and (p / "simulation.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside WorldNTN.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from config import SimulationConfig, GroundStation, RadioConfig, TriggerConfig, RewardConfig, ground_east_of, constellation_name
from orbit_io import discover_csvs
from dataset import run_dataset, link_budget_table, TrainingView
# Install dependencies in this kernel if needed:
# %pip install -r "{ROOT / 'requirements.txt'}"
SATELLITE_ROOT = ROOT.parent / "generate satellite"
print("Data generator:", ROOT)


## Select constellations and orbit CSV files

`amazon` maps automatically to `kuiper`. Each side may use one real CSV, or one real CSV plus one synthetic completion CSV. Do not combine both alternative synthetic scenarios. `None` automatically selects the latest **real all_states** CSV for that constellation and does not add synthetic data.

The upstream generator exports only visible CSVs by default. Set `write_all_states=true` in `generate satellite/scenario.json`, then rerun the required trackers. For handovers at second-scale resolution, also set the upstream `interval_sec` to 1–5 seconds. A 120-second orbit grid retains its original sampling resolution here.


In [ ]:
victim = "starlink"
aggressor = "amazon"

# Explicit examples: [Path("..._all_states.csv"), Path("...synthetic..._all_states.csv")]
VICTIM_CSVS = None
AGGRESSOR_CSVS = None
available = discover_csvs(SATELLITE_ROOT)
display(available)

def latest_real_csv(constellation):
    matching = available[(available.constellation == constellation_name(constellation))
                         & (available.coverage == "all_states")
                         & (available.source_type == "real_gp")
                         & (available.status == "complete")]
    return [Path(matching.iloc[-1].path)] if len(matching) else []

selected_victim = [Path(p) for p in VICTIM_CSVS] if VICTIM_CSVS is not None else latest_real_csv(victim)
selected_aggressor = [Path(p) for p in AGGRESSOR_CSVS] if AGGRESSOR_CSVS is not None else latest_real_csv(aggressor)
print("Victim inputs:", selected_victim)
print("Aggressor inputs:", selected_aggressor)


## Radio settings, ground stations, policies, and triggers

The recommended baseline transmit power is **10 W per active satellite across the 250 MHz channel**. This is channel power, not power spectral density or total spacecraft power. Transmit and receive gains initially use 30 dBi each, identically for both sides. The link budget is displayed below for adjustment.

By default, A's initial selection and B's policy use `longest_remaining`; A's handover policy uses `random`. Change it to `longest_remaining` or `max_snr` for comparisons. These alternatives use SNR predicted from the fixed orbit inputs without accessing future interference.

Assign 5 points for SINR < 0 dB, 3 for [0,3), 2 for [3,6), and 1 for ≥6. A sum ≥10 over the latest 5 samples triggers a handover at the next time step. Clear the old scores after a successful handover. These are research thresholds to calibrate against the observed distributions.


In [ ]:
ground_a = GroundStation(lat_deg=39.95697, lon_deg=-105.16033, height_m=1660.0)
ground_b = ground_east_of(ground_a, distance_m=1000.0)
radio = RadioConfig(
    frequency_hz=20e9, bandwidth_hz=250e6, tx_power_w=10.0,
    tx_gain_dbi=30.0, rx_gain_dbi=30.0,
    temperature_k=290.0, noise_figure_db=6.0, interference_enabled=True,
)
trigger = TriggerConfig(
    sinr_min_db=0.0, middle_start_db=3.0, highest_start_db=6.0,
    window_samples=5, trigger_points=10, bin_points=(5, 3, 2, 1),
    force_on_geometry_loss=False,
)
reward = RewardConfig(
    time_scale_s=60.0, dwell_weight=1.0, quality_weight=1.0,
    outage_weight=2.0, handover_cost=0.05, success_score_threshold=2.0,
)
cfg = SimulationConfig(
    victim=victim, aggressor=aggressor, ground_a=ground_a, ground_b=ground_b,
    radio=radio, trigger=trigger, reward=reward,
    min_snr_db=0.0, min_elevation_deg=25.0,
    aggressor_policy="longest_remaining",
    victim_initial_policy="longest_remaining", victim_handover_policy="random",
    random_seed=20261005, start_time_utc=None, max_steps=600,
    history_samples=10, prediction_samples=1, window_stride=1,
    split="train", scenario_group_id=None,
    allow_partial_catalog=False, allow_unverified_csv=False, assume_naive_utc=False,
).validate()
display(link_budget_table(radio))
print("B site:", ground_b)


## Generate the dataset

Each run creates a separate UTC-stamped directory and records input hashes, parameters, seed, code version, and quality statistics. The source orbit CSVs are preserved.

`public/observations.csv` records SINR/SNR/INR and the interference arrival direction for A's serving link. `public/geometry.npz` stores positions, velocities, and validity masks for every input A/B satellite. B's actual connections, private ground station, and policy truth are stored in `private/` and excluded from training features.


In [ ]:
RUN_GENERATION = True
run_directory = None
if not selected_victim or not selected_aggressor:
    print("No complete pair selected. Export all_states CSVs upstream or set VICTIM_CSVS / AGGRESSOR_CSVS, then rerun.")
elif RUN_GENERATION:
    run_directory = run_dataset(selected_victim, selected_aggressor, cfg, ROOT / "outputs")
    print("Dataset:", run_directory)


## Inspect triggers, actions, and handover scores

Handover execution success and subsequent link-quality success are separate outcomes. The final connection segment has `right_censored=True`; a short observed dwell caused by the CSV ending must not be labeled a failure. When no alternative satellite is available, preserve the current state and record `blocked_no_candidate` without claiming a successful handover.


In [ ]:
if run_directory is not None:
    display(json.loads((run_directory / "quality_report.json").read_text()))
    observations = pd.read_csv(run_directory / "public" / "observations.csv")
    actions = pd.read_csv(run_directory / "public" / "actions.csv")
    segments = pd.read_csv(run_directory / "labels" / "link_segments.csv")
    display(observations.head(12))
    display(actions[actions.action != "hold"].head(12))
    display(segments.head(12))
    display(observations.quality_points.value_counts(normalize=True).rename("fraction"))


## Training view: state history + current action → next observation

Use one-step prediction by default. Validate the action-conditioned prediction in the scenario diagram before extending to multi-step rollouts. `TrainingView` reads neither `private/` nor future handover scores. NaNs become zeros with accompanying masks; fill values must not be treated as actual 0 dB measurements.

Runs with the same underlying orbits and ground stations share a `scenario_group_id`, even with different random seeds or policies. Do not randomly split overlapping windows between training and testing. Scores in `link_segments.csv` are action-outcome labels and must not enter features available before the action.


In [ ]:
if run_directory is not None:
    training = TrainingView(run_directory)
    if len(training.windows):
        sample = training.sample(0)
        print("Feature names:", sample["inputs"]["feature_names"])
        print("History shape:", sample["inputs"]["history"].shape)
        print("A orbit history shape:", sample["inputs"]["a_position_m"].shape)
        print("B orbit history shape:", sample["inputs"]["b_position_m"].shape)
        print("Action:", sample["inputs"]["action"])
        print("Target:", sample["labels"])
    else:
        print("Too few samples for the configured history and prediction windows.")
    training.close()


## Repeated collection and comparisons

Use `replace(cfg, random_seed=...)` for repeated random exploration. Change `victim_handover_policy` to compare longest-duration and maximum-SNR choices. Set `radio=replace(radio, interference_enabled=False)` for a control run without interference. Keep the inputs, ground stations, timeline, grouping, B's trajectories, and B's connection rules consistent.

With constant gains, selecting another A satellite does not suppress B's interference through a change in receive direction. It changes the desired signal, dwell time, and action timing. This version supports validation of the data pipeline, timing, and policies; studying directional interference avoidance later requires angle-dependent gains.
